In [3]:
import importlib as il
import numpy as np
import more_itertools as mit
import scipy.sparse as sp
import scipy.sparse.linalg as spla
import scipy.linalg as spl
import linetimer as lt

import gurobipy as gp

import gurobi_utils as gu
import dikin_utils as du
import plot_utils as pu

import example_loader as el
import miplib_loader as ml
import jsplib_loader as jl
import knapsack_loader as kl

status_lookup = {getattr(gp.GRB.Status, k): k for k in gp.GRB.Status.__dir__() if "A" <= k[0] <= "Z"}

%matplotlib inline

In [4]:
def get_A_b_c_l_u(mdl: gp.Model):
    mdl.update()
    A = mdl.getA()
    b = np.array(mdl.getAttr("RHS")).reshape(-1, 1)
    c = np.array(mdl.getAttr("Obj"))
    l = np.array(mdl.getAttr("LB"))
    u = np.array(mdl.getAttr("UB"))
    return A, b, c, l , u

def sparse_null_space(A, tol=1e-7):
    import sparseqr as rz
    if not sp.issparse(A):
        raise TypeError("Input must be a sparse matrix.")
    
    m, n = A.shape

    # Compute rank-revealing QR (with column permutation)
    Q, _, E, rank = rz.qr(A, economy=False)
    nullity = n - rank

    # Early return if full rank
    if nullity == 0:
        return sp.csc_matrix((n, 0))

    # Apply inverse permutation to align with original columns
    E_inv = np.argsort(E)  # Inverse permutation

    # Extract nullity trailing columns (explicitly)
    Z = Q.tocsr()[E_inv, -nullity:]
    return Z

def get_x0(mdl: gp.Model):
    cpy = mdl.copy()
    cpy.params.LogToConsole = 0
    gu.relax_int_or_bin_to_continuous(cpy)
    cpy.optimize()
    if cpy.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Relaxation not optimal: {status_lookup[cpy.status]}")
    x0 = np.array([v.X for v in cpy.getVars()]).reshape((-1, 1))
    return x0

In [ ]:
il.reload(kl)
il.reload(gu)

def adaptive_null_space(A):
    """
    Computes null space with tolerance automatically adjusted based on condition number
    """
    # Compute SVD
    u, s, vh = spl.svd(A, full_matrices=True)
    
    # Calculate condition number (handle case where min singular value is 0)
    max_s = s[0]
    min_s = s[-1] if s[-1] > 0 else s[s > 0][-1]
    cond = max_s / min_s
    
    # Machine epsilon for float64
    eps = np.finfo(np.float64).eps
    
    # Adaptive tolerance
    tol = max(A.shape) * cond * eps
    print(f"Adaptive tolerance: {tol:.2e}, Condition number: {cond:.2e}")
    
    # Find null space basis vectors
    rank = (s >= tol).sum()
    null_space = vh[rank:].conj().T
    
    return null_space

instances = kl.generate(3, 2, 20, 1, 60, 400, equality=True, seed=43)
for model in instances:
    # assumptions on the model: all equality constraints, fully linear objective & constraints, all vars >= 0, maximizing
    x0 = get_x0(model)
    A = model.getA()
    b = np.array(model.getAttr("RHS")).reshape((-1, 1))
    m, n = A.shape
    assert n > m and np.allclose(A @ x0, b)
    # N = spl.null_space(A.todense(), check_finite=False, rcond=1e-7, lapack_driver='gesvd') # sparse_null_space(A)
    N = adaptive_null_space(A.todense())
    c = np.array(model.getAttr("Obj")).reshape((-1, 1))
    ub = np.array(model.getAttr("UB")).reshape((-1, 1)) # can't go without this?

    mdl2 = gp.Model(model.ModelName + "_tfrm")
    mdl2.params.IntegralityFocus = 1
    mdl2.params.IntFeasTol = 1e-8
    z = mdl2.addMVar((n - m, 1), lb=-gp.GRB.INFINITY, vtype='C', name='z')
    print(c.shape, x0.shape, A.shape, N.shape, z.shape)
    y = mdl2.addMVar((n, 1), ub=ub, vtype='I', name='y')
    # mdl2.setObjective(c.T @ x0 + c.T @ N @ z, gp.GRB.MAXIMIZE)
    mdl2.setObjective(c.T @ y, gp.GRB.MAXIMIZE)
    # mdl2.addConstr(N @ z >= -x0)
    mdl2.addConstr(x0 + N@z == y)
    mdl2.params.LogToConsole = 0
    with lt.CodeTimer("Transformed optimization time"):
        mdl2.optimize()
    if mdl2.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Transformation not optimal: {status_lookup[mdl2.status]}")
    x1 = y.X
    
    model.params.LogToConsole = 0
    with lt.CodeTimer("Original optimization time"):
        model.optimize()
    if model.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Original model not optimal: {status_lookup[model.status]}")
    x2 = np.array([v.X for v in model.getVars()]).reshape((-1, 1))
    
    print(f"Objective value: {mdl2.ObjVal}. Expected: {model.ObjVal}")
    print(f"Transformed solution: {x1.T}")
    print(f"Original solution: {x2.T}")
    

    


Set parameter LogToConsole to value 0


   Relaxed 20 variables on knapsack_2_20_0_copy
Adaptive tolerance: 1.56e-14, Condition number: 3.52e+00
Set parameter IntegralityFocus to value 1
Set parameter IntFeasTol to value 1e-08
(20, 1) (20, 1) (2, 20) (20, 18) (18, 1)
Set parameter LogToConsole to value 0
Code block 'Transformed optimization time' took: 2807.84753 ms
Set parameter LogToConsole to value 0
Code block 'Original optimization time' took: 3453.23283 ms
Objective value: 90760.0. Expected: 90760.0
Transformed solution: [[ 0.  0. 52.  0. 47.  0. 53. 13. 24.  1. 29.  0. 24.  0.  6. 43. 24. 37.
   0. 15.]]
Original solution: [[ 0.  0. 52.  0. 47.  0. 53. 13. 24.  1. 29.  0. 24.  0.  6. 43. 24. 37.
   0. 15.]]
Set parameter LogToConsole to value 0
   Relaxed 20 variables on knapsack_2_20_1_copy
Adaptive tolerance: 1.33e-14, Condition number: 3.00e+00
Set parameter IntegralityFocus to value 1
Set parameter IntFeasTol to value 1e-08
(20, 1) (20, 1) (2, 20) (20, 18) (18, 1)
Set parameter LogToConsole to value 0
Code block '

In [78]:
import hsnf

def solve_via_snf(A: np.ndarray, b: np.ndarray) -> np.ndarray:
    D, L, R = hsnf.smith_normal_form(A)
    m, n = A.shape
    rank = np.count_nonzero(np.diag(D))  # Number of non-zero diagonal entries in D
    
    # Step 1: Compute L b
    Lb = L @ b
    
    # Step 2: Solve D y = L b
    y_p = np.zeros((n, 1), dtype=int)  # Particular solution for y
    
    for i in range(rank):
        d_i = D[i, i]
        if d_i == 0:
            if Lb[i, 0] != 0:
                raise ValueError("No solution exists (inconsistent system)")
        else:
            if Lb[i, 0] % d_i != 0:
                raise ValueError("No integer solution exists (Lb not divisible by D[i,i])")
            y_p[i, 0] = Lb[i, 0] // d_i
    
    # Step 3: Compute x_p = R y_p
    x_p = R @ y_p

    assert np.allclose(A @ x_p, b), "x_p is not a solution to Ax = b"
    null_basis = R[:, rank:]
    # assert np.allclose(A @ N, 0), "invalid nulspace basis"

    return x_p, null_basis

instances = kl.generate(3, 2, 50, 5, 10, 1000, equality=True, seed=43)
for model in instances:
    model.params.LogToConsole = 0
    # assumptions on the model: all equality constraints, fully linear objective & constraints, all vars >= 0, maximizing
    A = model.getA().todense()
    b = np.array(model.getAttr("RHS")).reshape((-1, 1))
    c = np.array(model.getAttr("Obj")).reshape((-1, 1))
    ub = np.array(model.getAttr("UB")).reshape((-1, 1))

    with lt.CodeTimer("Original optimization time"):
        model.optimize()
    if model.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Original model not optimal: {status_lookup[model.status]}")
    print(f"Original objective value: {model.ObjVal}")

    xp, N = solve_via_snf(A, b)
    # now I have an integer null space and an integer starting solution (that may violate bounds)

    mdl2 = gp.Model(model.ModelName + "_tfrm")
    z = mdl2.addMVar((N.shape[1], 1), lb=-gp.GRB.INFINITY, vtype='I', name='z')
    mdl2.setObjective(c.T @ xp + c.T @ N @ z, gp.GRB.MAXIMIZE)
    mdl2.addConstr(xp + N@z >= 0)
    mdl2.addConstr(xp + N@z <= ub)
    # mdl2.params.NumericFocus = 3
    # mdl2.params.DualReductions = 0
    mdl2.params.LogToConsole = 0
    with lt.CodeTimer("Transformed optimization time"):
        mdl2.optimize()
    if mdl2.status != gp.GRB.Status.OPTIMAL:
        raise ValueError(f"Transformation not optimal: {status_lookup[mdl2.status]}")
    print("Objective value: ", mdl2.ObjVal)
    mdl2.dispose()
    print()


Set parameter LogToConsole to value 0
Code block 'Original optimization time' took: 9563.90258 ms
Original objective value: 153792.0
Set parameter LogToConsole to value 0
Code block 'Transformed optimization time' took: 19186.99249 ms
Objective value:  153792.0

Set parameter LogToConsole to value 0
Code block 'Original optimization time' took: 11701.96077 ms
Original objective value: 141044.0
Set parameter LogToConsole to value 0
Code block 'Transformed optimization time' took: 15112.18248 ms
Objective value:  141042.0

Set parameter LogToConsole to value 0
Code block 'Original optimization time' took: 6435.96904 ms
Original objective value: 145749.0
Set parameter LogToConsole to value 0
Code block 'Transformed optimization time' took: 3654.01887 ms
Objective value:  145748.0

